<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn">
<p>قفل‌گذاری انحصاری یک مکانیسم برای اطمینان از این است که فقط یک thread در هر زمان بتواند به یک بخش خاص از کد یا یک منبع دسترسی داشته باشد. این مکانیسم برای جلوگیری از شرایط رقابتی (Race Conditions) و اطمینان از ایمنی در threading (Thread Safety) در برنامه‌های چندنخی بسیار مهم است. در سی‌شارپ، سه ساختار اصلی برای قفل‌گذاری انحصاری وجود دارد:</p>
<ol start="1"><li><p><strong>دستور <code>lock</code></strong>: پرکاربردترین و ساده‌ترین مکانیزم قفل‌گذاری.</p></li><li><p><strong><code>Mutex</code></strong>: برای قفل‌گذاری در سطح چندین پردازش (قفل‌های سیستم‌wide).</p></li><li><p><strong><code>SpinLock</code></strong>: یک مکانیزم قفل‌گذاری سبک‌وزن که برای سناریوهای با همزمانی بالا بهینه‌سازی شده است.</p></li></ol>
</div>

### **The lock Statement**

In [ ]:
class ThreadUnsafe
{
    static int _val1 = 1, _val2 = 1;

    static void Go()
    {
        if (_val2 != 0) Console.WriteLine(_val1 / _val2);
        _val2 = 0;
    }
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn">
<h4>مشکل این کلاس:</h4>
<ul><li><p>این کلاس دو متغیر استاتیک به نام‌های <code>_val1</code> و <code>_val2</code> دارد.</p></li><li><p>متد <code>Go</code> بررسی می‌کند که آیا <code>_val2</code> برابر با صفر نیست. اگر نبود، مقدار <code>_val1 / _val2</code> را چاپ می‌کند و سپس <code>_val2</code> را به صفر تنظیم می‌کند.</p></li><li><p>اگر دو thread همزمان متد <code>Go</code> را فراخوانی کنند، ممکن است یک <strong>شرایط رقابتی</strong> (Race Condition) رخ دهد. برای مثال:</p><ol start="1"><li><p>Thread A بررسی می‌کند که <code>_val2</code> برابر با صفر نیست و وارد بلوک <code>if</code> می‌شود.</p></li><li><p>قبل از اینکه Thread A دستور <code>Console.WriteLine</code> را اجرا کند، Thread B مقدار <code>_val2</code> را به صفر تغییر می‌دهد.</p></li><li><p>حالا Thread A سعی می‌کند <code>_val1</code> را بر <code>_val2</code> تقسیم کند، اما چون <code>_val2</code> اکنون صفر است، یک <strong>خطای تقسیم بر صفر</strong> رخ می‌دهد.</p></li></ol></li></ul>
<h3>حل مشکل با استفاده از <code>lock</code></h3>
<p>برای رفع این مشکل، از دستور <code>lock</code> استفاده می‌شود تا اطمینان حاصل شود که فقط یک thread در هر زمان می‌تواند بخش بحرانی کد را اجرا کند. کلاس اصلاح‌شده به صورت زیر است:</p>
</div>

In [ ]:
class ThreadSafe
{
    static readonly object _locker = new object();
    static int _val1 = 1, _val2 = 1;

    static void Go()
    {
        lock (_locker)
        {
            if (_val2 != 0) Console.WriteLine(_val1 / _val2);
            _val2 = 0;
        }
    }
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn">
<h4>نحوه کار <code>lock</code>:</h4>
<ol start="1"><li><p>یک شیء همگام‌سازی به نام <code>_locker</code> ایجاد می‌شود. این شیء برای قفل‌گذاری استفاده می‌شود.</p></li><li><p>وقتی یک thread وارد بلوک <code>lock</code> می‌شود، قفل انحصاری روی <code>_locker</code> می‌گیرد.</p></li><li><p>اگر thread دیگری سعی کند وارد بلوک <code>lock</code> شود در حالی که قفل در اختیار thread دیگری است، منتظر می‌ماند (block می‌شود) تا قفل آزاد شود.</p></li><li><p>وقتی thread اول از بلوک <code>lock</code> خارج می‌شود، قفل آزاد می‌شود و thread بعدی در صف می‌تواند قفل را بگیرد.</p></li></ol>
<p>این مکانیزم اطمینان می‌دهد که کد داخل بلوک <code>lock</code> فقط توسط یک thread در هر زمان اجرا می‌شود و از شرایط رقابتی جلوگیری می‌کند.</p>
<h3>نکات کلیدی:</h3>
<ol start="1"><li><p><strong>قفل انحصاری</strong>: فقط یک thread می‌تواند در هر زمان قفل را در اختیار بگیرد. سایر threadها باید منتظر بمانند تا قفل آزاد شود.</p></li><li><p><strong>صف threadها</strong>: اگر چندین thread برای گرفتن قفل رقابت کنند، در یک صف قرار می‌گیرند و به ترتیب اولویت (اولین ورود، اولین سرویس) قفل را دریافت می‌کنند.</p></li><li><p><strong>دسترسی سریالی</strong>: قفل‌های انحصاری دسترسی سریالی به کد یا منابع محافظت‌شده را تضمین می‌کنند. این یعنی دسترسی یک thread نمی‌تواند با دسترسی thread دیگری هم‌پوشانی داشته باشد.</p></li><li><p><strong>محافظت از منابع</strong>: در این مثال، دستور <code>lock</code> از منطق داخل متد <code>Go</code> و همچنین از فیلدهای <code>_val1</code> و <code>_val2</code> محافظت می‌کند.</p></li></ol>
</div>

### **Monitor.Enter and Monitor.Exit**

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn">
<p>این متن به بررسی جزئیات پشت‌پرده‌ی دستور <code>lock</code> در سی‌شارپ می‌پردازد و نشان می‌دهد که <code>lock</code> در واقع یک میانبر سینتکسی (Syntactic Sugar) برای استفاده از متدهای <code>Monitor.Enter</code> و <code>Monitor.Exit</code> است. همچنین، متن به برخی از مشکلات احتمالی و راه‌حل‌های آن‌ها اشاره می‌کند.</p>
<h3>دستور <code>lock</code> و معادل آن با <code>Monitor.Enter</code> و <code>Monitor.Exit</code></h3>
<p>در سی‌شارپ، دستور <code>lock</code> به‌صورت زیر استفاده می‌شود:</p>
</div>

In [ ]:
lock (_locker)
{
    // کد بحرانی
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn">
<p>اما در واقع، این دستور یک میانبر سینتکسی برای کد زیر است:</p>
</div>

In [ ]:
Monitor.Enter(_locker);
try
{
    // کد بحرانی
}
finally
{
    Monitor.Exit(_locker);
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn">
<h4>توضیح:</h4>
<ol start="1"><li><p><strong><code>Monitor.Enter</code></strong>: این متد قفل را روی شیء <code>_locker</code> می‌گیرد. اگر قفل در اختیار thread دیگری باشد، thread فعلی منتظر می‌ماند تا قفل آزاد شود.</p></li><li><p><strong><code>try/finally</code></strong>: کد بحرانی داخل بلوک <code>try</code> اجرا می‌شود. بلوک <code>finally</code> تضمین می‌کند که حتی اگر یک استثنا (Exception) رخ دهد، قفل حتماً آزاد می‌شود.</p></li><li><p><strong><code>Monitor.Exit</code></strong>: این متد قفل را آزاد می‌کند تا threadهای دیگر بتوانند آن را بگیرند.</p></li></ol>
<h3>مشکل احتمالی در استفاده از <code>Monitor.Enter</code> و <code>Monitor.Exit</code></h3>
<p>متن به یک مشکل ظریف در استفاده از <code>Monitor.Enter</code> و <code>Monitor.Exit</code> اشاره می‌کند. اگر بین فراخوانی <code>Monitor.Enter</code> و بلوک <code>try</code> یک استثنا رخ دهد (مثلاً به دلیل <code>OutOfMemoryException</code> یا در .NET Framework، اگر thread لغو شود)، ممکن است قفل گرفته شود اما هرگز آزاد نشود. این وضعیت منجر به <strong>نشت قفل</strong> (Leaked Lock) می‌شود.</p>
</div>

In [ ]:
Monitor.Enter(_locker); // اگر اینجا استثنا رخ دهد، قفل ممکن است گرفته شود اما آزاد نشود.
try
{
    // کد بحرانی
}
finally
{
    Monitor.Exit(_locker);
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn">
<h3>راه‌حل: استفاده از <code>Monitor.Enter</code> با پارامتر <code>lockTaken</code></h3>
<p>برای جلوگیری از نشت قفل، <code>Monitor.Enter</code> یک اورلود (Overload) دارد که یک پارامتر <code>ref bool lockTaken</code> می‌گیرد. این اورلود تضمین می‌کند که اگر <code>Monitor.Enter</code> با استثنا مواجه شود، مقدار <code>lockTaken</code> برابر با <code>false</code> خواهد بود و قفل گرفته نخواهد شد.</p>
</div>

In [ ]:
bool lockTaken = false;
try
{
    Monitor.Enter(_locker, ref lockTaken);
    // کد بحرانی
}
finally
{
    if (lockTaken)
        Monitor.Exit(_locker);
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn">
<h4>توضیح:</h4>
<ol start="1"><li><p><strong><code>lockTaken</code></strong>: این پارامتر نشان می‌دهد که آیا قفل با موفقیت گرفته شده است یا خیر.</p></li><li><p>اگر <code>Monitor.Enter</code> با استثنا مواجه شود، <code>lockTaken</code> برابر با <code>false</code> خواهد بود و در بلوک <code>finally</code> قفل آزاد نخواهد شد.</p></li><li><p>اگر قفل با موفقیت گرفته شود، <code>lockTaken</code> برابر با <code>true</code> خواهد بود و در بلوک <code>finally</code> قفل آزاد می‌شود.</p></li></ol>

<h3>متد <code>TryEnter</code></h3>
<p><code>Monitor</code> همچنین متدی به نام <code>TryEnter</code> ارائه می‌دهد که امکان تعیین زمان انتظار (Timeout) برای گرفتن قفل را فراهم می‌کند. این متد برخلاف <code>Enter</code>، اگر نتواند قفل را بگیرد، به جای منتظر ماندن، بلافاصله یا پس از مدت زمان مشخصی بازمی‌گردد.</p>
</div>

In [ ]:
bool lockTaken = false;
try
{
    Monitor.TryEnter(_locker, 500, ref lockTaken); // 500 میلی‌ثانیه منتظر می‌ماند.
    if (lockTaken)
    {
        // کد بحرانی
    }
    else
    {
        // اگر قفل گرفته نشد، این بخش اجرا می‌شود.
    }
}
finally
{
    if (lockTaken)
        Monitor.Exit(_locker);
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn">
<h4>توضیح:</h4>
<ol start="1"><li><p><strong>پارامتر زمان انتظار</strong>: می‌توانید زمان انتظار را به صورت میلی‌ثانیه یا <code>TimeSpan</code> مشخص کنید.</p></li><li><p><strong>مقدار بازگشتی</strong>: اگر قفل گرفته شود، <code>true</code> برمی‌گردد. اگر زمان انتظار به پایان برسد و قفل گرفته نشود، <code>false</code> برمی‌گردد.</p></li><li><p><strong>اورلود بدون پارامتر</strong>: اگر <code>TryEnter</code> بدون پارامتر زمان انتظار فراخوانی شود، بلافاصله بازمی‌گردد و اگر قفل در دسترس نباشد، <code>false</code> برمی‌گردد.</p></li></ol>
</div>

### **Choosing the Synchronization Object**

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn">
<p>این متن به موضوع انتخاب شیء مناسب برای همگام‌سازی (Synchronization Object) در برنامه‌های چندنخی (Multi-threaded) می‌پردازد. انتخاب شیء همگام‌سازی مناسب برای قفل‌گذاری (<code>lock</code>) بسیار مهم است، زیرا این شیء تعیین می‌کند که چگونه threadها به منابع مشترک دسترسی پیدا کنند.</p>
<h3>قوانین انتخاب شیء همگام‌سازی</h3>
<ol start="1"><li><p><strong>نوع شیء</strong>: شیء همگام‌سازی باید یک <strong>نوع ارجاعی</strong> (Reference Type) باشد. این یک قانون سخت‌گیرانه است و استفاده از انواع مقداری (Value Types) مانند <code>int</code> یا <code>struct</code> باعث ایجاد خطای کامپایل می‌شود.</p></li><li><p><strong>دسترسی به شیء</strong>: شیء همگام‌سازی باید برای تمام threadهایی که نیاز به همگام‌سازی دارند، قابل مشاهده باشد. این شیء معمولاً به عنوان یک فیلد <strong>خصوصی</strong> (private) یا <strong>استاتیک</strong> (static) تعریف می‌شود تا منطق قفل‌گذاری کپسوله شود و از تداخل با سایر بخش‌های کد جلوگیری شود.</p></li></ol>
<h3>مثال‌هایی از انتخاب شیء همگام‌سازی</h3>
<h4>۱. استفاده از یک فیلد اختصاصی برای قفل‌گذاری</h4>
<p>در این مثال، یک فیلد اختصاصی به نام <code>_locker</code> برای قفل‌گذاری استفاده می‌شود:</p>
</div>

In [ ]:
class ThreadSafe
{
    private readonly object _locker = new object();
    private List<string> _list = new List<string>();

    void Test()
    {
        lock (_locker)
        {
            _list.Add("Item 1");
            // سایر عملیات‌ها
        }
    }
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn">
<ul><li><p><strong>مزیت</strong>: استفاده از یک فیلد اختصاصی مانند <code>_locker</code> به شما امکان کنترل دقیق‌تر روی محدوده و دانه‌بندی (Granularity) قفل‌گذاری را می‌دهد.</p></li><li><p><strong>کپسوله‌سازی</strong>: این روش منطق قفل‌گذاری را کپسوله می‌کند و از تداخل با سایر بخش‌های کد جلوگیری می‌کند.</p></li></ul>
<h4>۲. استفاده از شیء محافظت‌شده به عنوان شیء همگام‌سازی</h4>
<p>در برخی موارد، می‌توانید از همان شیء‌ای که محافظت می‌کنید به عنوان شیء همگام‌سازی استفاده کنید. به عنوان مثال:</p>
</div>

In [ ]:
class ThreadSafe
{
    private List<string> _list = new List<string>();

    void Test()
    {
        lock (_list)
        {
            _list.Add("Item 1");
            // سایر عملیات‌ها
        }
    }
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn">
<ul><li><p><strong>مزیت</strong>: این روش ساده است و نیاز به تعریف یک فیلد اضافی برای قفل‌گذاری ندارد.</p></li><li><p><strong>معایب</strong>: اگر از این روش به‌طور گسترده استفاده شود، ممکن است کنترل دقیق روی قفل‌گذاری از دست برود و منجر به مشکلاتی مانند <strong>Deadlock</strong> یا <strong>Blocking بیش از حد</strong> شود.</p></li></ul>
<h4>۳. استفاده از <code>this</code> به عنوان شیء همگام‌سازی</h4>
<p>شما می‌توانید از شیء فعلی (<code>this</code>) به عنوان شیء همگام‌سازی استفاده کنید:</p>
</div>

In [ ]:
class ThreadSafe
{
    void Test()
    {
        lock (this)
        {
            // کد بحرانی
        }
    }
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn">
<ul><li><p><strong>مزیت</strong>: ساده و سریع است.</p></li><li><p><strong>معایب</strong>:</p><ul><li><p>این روش منطق قفل‌گذاری را کپسوله نمی‌کند و ممکن است سایر بخش‌های کد نیز از <code>this</code> برای قفل‌گذاری استفاده کنند.</p></li><li><p>این موضوع می‌تواند منجر به <strong>Deadlock</strong> یا <strong>Blocking بیش از حد</strong> شود، زیرا کنترل دقیقی روی قفل‌گذاری وجود ندارد.</p></li></ul></li></ul>
<h4>۴. استفاده از <code>typeof</code> برای قفل‌گذاری روی اعضای استاتیک</h4>
<p>اگر نیاز به محافظت از اعضای استاتیک دارید، می‌توانید از <code>typeof</code> برای قفل‌گذاری استفاده کنید:</p>
</div>

In [ ]:
class Widget
{
    static int _staticValue = 0;

    void Test()
    {
        lock (typeof(Widget))
        {
            _staticValue++;
            // سایر عملیات‌ها
        }
    }
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn">
<ul><li><p><strong>مزیت</strong>: این روش برای محافظت از اعضای استاتیک مفید است.</p></li><li><p><strong>معایب</strong>:</p><ul><li><p>قفل‌گذاری روی <code>typeof</code> می‌تواند منجر به <strong>Blocking گسترده</strong> شود، زیرا تمام threadهایی که از این نوع استفاده می‌کنند، ممکن است مسدود شوند.</p></li><li><p>این روش نیز منطق قفل‌گذاری را کپسوله نمی‌کند.</p></li></ul></li></ul>
<h4>۵. استفاده از متغیرهای محلی در لامبدا یا متدهای ناشناس</h4>
<p>شما می‌توانید از متغیرهای محلی که توسط لامبدا یا متدهای ناشناس (Anonymous Methods) ضبط شده‌اند، به عنوان شیء همگام‌سازی استفاده کنید. به عنوان مثال:</p>
</div>

In [ ]:
void Test()
{
    object localLocker = new object();
    lock (localLocker)
    {
        // کد بحرانی
    }
}

<div dir="rtl" style="margin:auto; width:90%; font-family:vazirmatn">
<ul><li><p><strong>مزیت</strong>: این روش برای قفل‌گذاری در محدوده‌های کوچک و موقت مفید است.</p></li><li><p><strong>معایب</strong>: این روش برای قفل‌گذاری در سطح گسترده مناسب نیست، زیرا کنترل دقیقی روی قفل‌گذاری وجود ندارد.</p></li></ul>
</div>